# 05 · Escenarios operativos (20) y elección del modelo

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

El enunciado pide evaluar *"al menos 20 escenarios operativos con estados iniciales reproducibles"* y reportar el *"porcentaje de escenarios completados correctamente"* y los *"incumplimientos de restricciones"*. Este notebook los ejecuta en **texto**. Los 6 de voz se repiten con audio en el notebook 06.

### Cómo se construye cada escenario (`src/asistente_compras/scenarios.py`)
| Elemento | Qué es |
|---|---|
| **Estado inicial** | Tienda restaurada con la semilla + stock y precios fijados para los productos que devuelve la búsqueda de ciertas consultas (`Setup`). Se registra su huella: es idéntico en cada ejecución |
| **Turnos** | Lo que dice el usuario, en orden |
| **Eventos** | Cambios externos entre turnos: agotar lo mostrado, bajar el stock del carrito, subir un precio |
| **Verificaciones** | Comprobaciones automáticas sobre el pedido final, la traza de herramientas y las respuestas |
| **Violaciones** | Conteo genérico: presupuesto excedido, cantidad incorrecta, pedido duplicado, compra sin confirmación |

Un escenario es **correcto** si pasan todas sus verificaciones y no hay ninguna violación. Cada ejecución se guarda como JSON en `data/scenarios/<modelo>/<ID>.json`, con turnos, herramientas, respuestas, bitácora de la tienda y estado final.

> **Disciplina:** los escenarios se usaron para detectar fallos del *sistema* (por ejemplo, el agente no cotizaba) y corregirlos con reglas de código generales. No se ajustaron prompts a frases concretas de los escenarios.

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = "rag,llm"
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

In [2]:
import pandas as pd

from asistente_compras.config import get_paths
from asistente_compras.evaluation import timed
from asistente_compras.llm import free_gpu, make_llm
from asistente_compras.retrieval import build_system_retriever
from asistente_compras.scenarios import SCENARIOS, run_all

# Definir los modelos a comparar. En la Mac se usó solo el mejor (qwen3.5:9b) por tiempo;
# en Colab, comparar los candidatos para justificar la elección.
MODELOS = ["qwen3.5:9b"] if not EN_COLAB else ["qwen2.5:7b", "qwen3:8b", "qwen3.5:9b"]
SUFIJO = "_colab" if EN_COLAB else ""

paths = get_paths().ensure()
catalog = pd.read_parquet(paths.processed / "catalog.parquet")
retriever, t_ret = timed(build_system_retriever, catalog, paths.index)
print(f"Recuperador listo en {t_ret:.1f} s · modelos a evaluar: {MODELOS}")

Recuperador listo en 73.0 s · modelos a evaluar: ['qwen2.5:7b', 'qwen3:8b', 'qwen3.5:9b']


## 1. Los 20 escenarios

In [3]:
pd.DataFrame([{"id": s.id, "escenario": s.title, "categoría": s.category,
               "voz": "🎙" if s.voice else "", "turnos": len(s.turns),
               "eventos": ", ".join(e.kind for e in s.events) or "—",
               "verificaciones": len(s.checks)} for s in SCENARIOS]).set_index("id")

,escenario,categoría,voz,turnos,eventos,verificaciones
id,,,,,,
V1,Compra que cumple las restricciones,compra_ok,🎙,3,—,4
V2,Solicitud ambigua que requiere aclaración,ambigua,🎙,4,—,4
V3,Cambio de cantidad conservando el presupuesto,cambio,🎙,4,—,4
V4,Producto agotado: buscar alternativa,agotado,🎙,4,zero_shown,4
V5,Corrección de la transcripción antes de comprar,correccion,🎙,4,—,3
V6,Cambio de stock entre cotización y confirmació...,stock_cambia,🎙,6,reduce_cart_stock,5
T07,Compra simple sin presupuesto,compra_ok,,3,—,3
T08,Presupuesto insuficiente: no hay solución,presupuesto,,1,—,3
T09,Comparar dos alternativas,comparar,,2,—,3


## 2. Ejecución

Cada escenario parte de su estado inicial (tienda restaurada + `Setup`). En la Mac, cada modelo tarda unos 15–20 min en completar los 20.

In [4]:
resultados = {}
for m in MODELOS:
    print(f"\n════════ {m}")
    llm = make_llm(m)
    # En Colab, usar una carpeta propia (`_colab`) para no mezclar con los resultados de la Mac.
    out = paths.root / "scenarios" / (m.replace(":", "_").replace(".", "") + SUFIJO)
    resultados[m] = run_all(llm, retriever, catalog, paths.root / "store" / "escenario.db", out)
    resultados[m].to_csv(out / "resumen.csv", index=False)
    del llm
    free_gpu()  # liberar la GPU antes de cargar el siguiente modelo


════════ qwen2.5:7b


✅ V1   Compra que cumple las restricciones                     checks 4/4 · violaciones 0 · 5.3 s/turno
✅ V2   Solicitud ambigua que requiere aclaración               checks 4/4 · violaciones 0 · 4.5 s/turno
❌ V3   Cambio de cantidad conservando el presupuesto           checks 1/4 · violaciones 0 · 11.5 s/turno
        ↳ pedidos == 1: pedidos=0
        ↳ pedido contiene ('botella',): faltan ['botella']
        ↳ unidades pedidas == 5: unidades=0
❌ V4   Producto agotado: buscar alternativa                    checks 2/4 · violaciones 0 · 7.0 s/turno
        ↳ pedidos == 1: pedidos=0
        ↳ pedido contiene ('rat[oó]n|mouse',): faltan ['rat[oó]n|mouse']
❌ V5   Corrección de la transcripción antes de comprar         checks 0/3 · violaciones 0 · 4.4 s/turno
        ↳ pedidos == 1: pedidos=0
        ↳ pedido contiene ('l[aá]piz|l[aá]pices',): faltan ['l[aá]piz|l[aá]pices']
        ↳ unidades pedidas == 15: unidades=0
❌ V6   Cambio de stock entre cotización y confirmación; confir checks 2/5

✅ V1   Compra que cumple las restricciones                     checks 4/4 · violaciones 0 · 7.3 s/turno
❌ V2   Solicitud ambigua que requiere aclaración               checks 3/4 · violaciones 0 · 7.4 s/turno
        ↳ pedido contiene ('funda', 'iphone'): faltan ['iphone']
❌ V3   Cambio de cantidad conservando el presupuesto           checks 3/4 · violaciones 0 · 7.4 s/turno
        ↳ pedido contiene ('botella',): faltan ['botella']
❌ V4   Producto agotado: buscar alternativa                    checks 1/4 · violaciones 0 · 6.0 s/turno
        ↳ busca alternativas tras «out_of_stock»: herramientas tras el error: ['actualizar_carrito', 'consultar_precio_stock']
        ↳ pedidos == 1: pedidos=0
        ↳ pedido contiene ('rat[oó]n|mouse',): faltan ['rat[oó]n|mouse']
✅ V5   Corrección de la transcripción antes de comprar         checks 3/3 · violaciones 0 · 7.3 s/turno
❌ V6   Cambio de stock entre cotización y confirmación; confir checks 4/5 · violaciones 1 · 9.1 s/turno
        ↳ unidades

✅ V1   Compra que cumple las restricciones                     checks 4/4 · violaciones 0 · 29.0 s/turno
✅ V2   Solicitud ambigua que requiere aclaración               checks 4/4 · violaciones 0 · 10.4 s/turno
❌ V3   Cambio de cantidad conservando el presupuesto           checks 3/4 · violaciones 0 · 11.5 s/turno
        ↳ pedido contiene ('botella',): faltan ['botella']
❌ V4   Producto agotado: buscar alternativa                    checks 1/4 · violaciones 0 · 11.8 s/turno
        ↳ busca alternativas tras «out_of_stock»: herramientas tras el error: ['actualizar_carrito', 'consultar_precio_stock', 'actualizar_carrito', 'consultar_precio_stock']
        ↳ pedidos == 1: pedidos=0
        ↳ pedido contiene ('rat[oó]n|mouse',): faltan ['rat[oó]n|mouse']
✅ V5   Corrección de la transcripción antes de comprar         checks 3/3 · violaciones 0 · 10.6 s/turno
❌ V6   Cambio de stock entre cotización y confirmación; confir checks 4/5 · violaciones 0 · 9.2 s/turno
        ↳ se detectó «stock_ch

## 3. Resultados

In [5]:
def resumen(df):
    return pd.Series({
        "escenarios correctos": f"{df.passed.sum()}/{len(df)} ({df.passed.mean():.0%})",
        "  de voz (texto)": f"{df[df.voice].passed.sum()}/{df.voice.sum()}",
        "verificaciones OK": f"{df.checks_ok.sum()}/{df.checks_total.sum()}",
        "violaciones de restricciones": int(df.violations.sum()),
        "correcciones pedidas al LLM": int(df.guard_events.sum()),
        "respuestas sin respaldo": int(df.unsupported_claims.sum()),
        "llamadas al LLM por escenario": round(df.llm_calls.mean(), 1),
        "segundos por turno (media)": round(df.seconds_per_turn.mean(), 1),
    })

pd.DataFrame({m: resumen(df) for m, df in resultados.items()})

,qwen2.5:7b,qwen3:8b,qwen3.5:9b
escenarios correctos,8/20 (40%),14/20 (70%),16/20 (80%)
de voz (texto),2/6,2/6,3/6
verificaciones OK,41/63,54/63,57/63
violaciones de restricciones,1,1,0
correcciones pedidas al LLM,9,4,10
respuestas sin respaldo,0,1,3
llamadas al LLM por escenario,6.1,6.8,7.2
segundos por turno (media),6.2,8.2,11.6


In [6]:
# Contar las violaciones por tipo: deben ser 0 porque las restricciones las aplica el código.
cols = [c for c in next(iter(resultados.values())).columns if c.startswith("v_")]
pd.DataFrame({m: df[cols].sum() for m, df in resultados.items()}).rename(
    index=lambda c: c[2:].replace("_", " "))

,qwen2.5:7b,qwen3:8b,qwen3.5:9b
presupuesto excedido,0,0,0
cantidad incorrecta,1,1,0
pedido duplicado,0,0,0
compra sin confirmacion,0,0,0
stock negativo,0,0,0


In [7]:
# Ver el detalle por escenario (✅/❌) para cada modelo.
det = pd.DataFrame({m: df.set_index("id").passed.map({True: "✅", False: "❌"})
                    for m, df in resultados.items()})
det.insert(0, "escenario", [s.title for s in SCENARIOS])
det

,escenario,qwen2.5:7b,qwen3:8b,qwen3.5:9b
id,,,,
V1,Compra que cumple las restricciones,✅,✅,✅
V2,Solicitud ambigua que requiere aclaración,✅,❌,✅
V3,Cambio de cantidad conservando el presupuesto,❌,❌,❌
V4,Producto agotado: buscar alternativa,❌,❌,❌
V5,Corrección de la transcripción antes de comprar,❌,✅,✅
V6,Cambio de stock entre cotización y confirmació...,❌,❌,❌
T07,Compra simple sin presupuesto,✅,✅,✅
T08,Presupuesto insuficiente: no hay solución,✅,✅,✅
T09,Comparar dos alternativas,❌,✅,✅


In [8]:
# Listar las verificaciones que fallaron: insumo para el análisis de errores.
for m, df in resultados.items():
    print(f"══ {m}")
    for _, r in df[~df.passed].iterrows():
        print(f"❌ {r.id} {r.title}")
        for f in r.failed_checks:
            print(f"     ↳ {f}")

══ qwen2.5:7b
❌ V3 Cambio de cantidad conservando el presupuesto
     ↳ pedidos == 1: pedidos=0
     ↳ pedido contiene ('botella',): faltan ['botella']
     ↳ unidades pedidas == 5: unidades=0
❌ V4 Producto agotado: buscar alternativa
     ↳ pedidos == 1: pedidos=0
     ↳ pedido contiene ('rat[oó]n|mouse',): faltan ['rat[oó]n|mouse']
❌ V5 Corrección de la transcripción antes de comprar
     ↳ pedidos == 1: pedidos=0
     ↳ pedido contiene ('l[aá]piz|l[aá]pices',): faltan ['l[aá]piz|l[aá]pices']
     ↳ unidades pedidas == 15: unidades=0
❌ V6 Cambio de stock entre cotización y confirmación; confirmación repetida
     ↳ pedidos == 1: pedidos=0
     ↳ unidades pedidas == 2: unidades=0
     ↳ pedido contiene ('mochila',): faltan ['mochila']
❌ T09 Comparar dos alternativas
     ↳ llamó ver_detalle en turno 2: herramientas=[]
❌ T10 Atributo no verificable: no inventar
     ↳ respuesta del turno -1 ~ /no (se )?(indica|especifica|aparece|figura|consta|menciona|detalla)|desconoc|no tengo (esa |e

## 4. Traza completa de un escenario

V6 (cambio de stock entre cotización y confirmación, y confirmación repetida) para el último modelo evaluado. Muestra las herramientas, las reglas de código (⚙) y el estado de la tienda turno a turno.

In [9]:
import json

m = MODELOS[-1]
d = json.loads((paths.root / "scenarios" / (m.replace(":", "_").replace(".", "") + SUFIJO) / "V6.json").read_text())
for t, snap in zip(d["turns"], d["snapshots"]):
    print(f"👤 {t['user']}")
    if t["auto_slots"]:
        print(f"   ⚙ {t['auto_slots']}")
    for c in t["tool_calls"]:
        err = "" if c["ok"] else f"  ✗ {[e.get('code') for e in c['result'].get('errors', [])]}"
        print(f"   🔧 {c['name']}({json.dumps(c['args'], ensure_ascii=False)}){err}")
    print(f"🤖 {t['response']}")
    print(f"   📦 pedidos={snap['orders']} · cotizaciones={snap['quotes']} · líneas carrito={snap['cart_lines']}\n")
print("Evaluación:", {k: d["evaluation"][k] for k in ("passed", "checks_ok", "checks_total", "violations")})

   📦 pedidos=0 · cotizaciones=0 · líneas carrito=0
   📦 pedidos=0 · cotizaciones=0 · líneas carrito=0
   📦 pedidos=0 · cotizaciones=0 · líneas carrito=0
   📦 pedidos=0 · cotizaciones=0 · líneas carrito=0
   📦 pedidos=0 · cotizaciones=1 · líneas carrito=1
   📦 pedidos=1 · cotizaciones=1 · líneas carrito=0
Evaluación: {'passed': False, 'checks_ok': 4, 'checks_total': 5, 'violations': 0}


### Resumen

- **Garantías de código:** con cualquier modelo, 0 violaciones de presupuesto, cantidad, stock, duplicados o compra sin confirmación.
- **Calidad conversacional:** depende del modelo; la tabla de la sección 3 justifica la elección.
- **Siguiente:** `06_voz.ipynb`, con los 6 escenarios con audio, la repetición con la transcripción correcta, las métricas de voz y las latencias.